In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import time

In [2]:
# Read the dataset
df = pd.read_csv('ml_features_and_labels.csv')

# Use the provided split column to create train/test sets and exclude metadata
# columns that would leak information about the label.
deterministic_leak_cols = [
    "e3_cert_parsed",
    "e1_alg_suite_Unknown(0x11eb)",
    "e1_alg_suite_Unknown(0x11ec)",
    "e1_alg_suite_nan",
    "e3_sig_alg_Unknown(0x0028)",
    "e9_conn_outcome_Failure",
    "e9_conn_outcome_Incomplete",
    "e8_proto_context_TLSv1.2_or_older",
    "e1b_tls_version_SSL3.0",
    "e1b_ciphersuite_2",
    "e1b_ciphersuite_54",
    "e2b_tls_version_SSL3.0",
    "e2b_ciphersuite_2",
    "e2b_ciphersuite_54",
]
print(len(deterministic_leak_cols))

X_train = df.loc[df['split'] == 'train', deterministic_leak_cols]
y_train = df.loc[df['split'] == 'train', 'label']

X_test = df.loc[df['split'] == 'test', deterministic_leak_cols]
y_test = df.loc[df['split'] == 'test', 'label']

14


In [3]:
# Initialize the model
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    enable_categorical=True,
)

start_time = time.perf_counter()

# Train the model
model.fit(X_train, y_train)

end_time = time.perf_counter()
print(f"Training time: {end_time - start_time:.2f} seconds")

Training time: 0.09 seconds


In [4]:
start_time = time.perf_counter()
# Predict on the held-out test split
y_pred = model.predict(X_test)
end_time = time.perf_counter()
print(f"Prediction time: {end_time - start_time:.2f} seconds")

Prediction time: 0.00 seconds


In [5]:
print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9410147463134216
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      7000
           1       1.00      0.53      0.69      1002

    accuracy                           0.94      8002
   macro avg       0.97      0.76      0.83      8002
weighted avg       0.94      0.94      0.93      8002

